# Lesson 16 Lab — Tensor Cores and Dtype Semantics

**Puzzle:** When input dtype, accumulator precision, throughput, and error change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates input dtype, accumulator precision, throughput, and error and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

tl.dot can lower to target matrix instructions when dtype, shape, and backend permit it. Input storage, multiply precision, accumulator type, and output cast are separate choices. A throughput table is incomplete without error measured against a declared reference.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["input dtype, accumulator precision, throughput, and error"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Combining FP16, BF16, TF32, and FP8 under one 'Tensor Core' row compares different numerical contracts.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 16
LESSON_TITLE = 'Tensor Cores and Dtype Semantics'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260829
}


## 5. Freeze the experiment

**Experiment:** Compare BF16 Triton and torch.mm paths, retaining dtype, latency, throughput, and FP32-comparison error.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.025087999179959297,
  "secondary": 0.015456000342965126,
  "max_abs_error": 0.0,
  "passed": true,
  "details": {
    "triton_tflops": 10.699755451779136,
    "library_tflops": 17.36771804111532,
    "dtype": "torch.bfloat16",
    "fused_relu": false,
    "triton_samples_ms": [
      0.03929600119590759,
      0.03129599988460541,
      0.027648000046610832,
      0.025631999596953392,
      0.025728000327944756,
      0.029152000322937965,
      0.02550400048494339,
      0.025087999179959297,
      0.02412799932062626,
      0.024191999807953835,
      0.022016000002622604,
      0.02239999920129776,
      0.023520000278949738,
      0.02316799946129322,
      0.021824000403285027
    ],
    "library_samples_ms": [
      0.017216000705957413,
      0.016287999227643013,
      0.015263999812304974,
      0.01587199978530407,
      0.014976000413298607,
      0.015039999969303608,
      0.01568000018596649,
      0.0163199994713068,
      0.014688000082969666,
      0.

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Triton BF16 median | 0.0251 ms |
| Library BF16 median | 0.0155 ms |
| Maximum absolute error | 0.000e+00 |
| Acceptance gate | true |


## 8. Explain without overclaiming

BF16 Triton and library GEMM are reported separately: 0.0251 and 0.0155 ms, with FP32-comparison max error 0.000e+00.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Choose dtype from an application error envelope first, then optimize the eligible matrix path.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 16,
  "title": "Tensor Cores and Dtype Semantics",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260829
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.025087999179959297,
    "secondary": 0.015456000342965126,
    "max_abs_error": 0.0,
    "passed": true,
    "details": {
      "triton_tflops": 10.699755451779136,
      "library_tflops": 17.36771804111532,
      "dtype": "torch.bfloat16",
      "fused_relu": false,
      "triton_samples_ms": [
        0.03929600119590759,
        0.03129599988460541,
        0.027648000046610832,
        0.025631999596953392,
        0.025728000327944756,
        0.029152000322937965,
        0.02550400048494339,
        0.025087999179959297,
        0.02412799932062626,
       

## 10. Make the bounded decision

> Choose dtype from an application error envelope first, then optimize the eligible matrix path.

**Failure analysis:** Combining FP16, BF16, TF32, and FP8 under one 'Tensor Core' row compares different numerical contracts.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
